# NSGA-III Baseline — Without-Reliability Group

Scheduling and discrete-time simulation definitions for the corresponding experimental comparison group. Generated outputs are written outside the notebook.

In [ ]:
# Imports
import gym
from gym import spaces
import pandas as pd
import numpy as np
import random
import math
from collections import deque
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d 
import os
from itertools import zip_longest
from scipy.stats import truncnorm
import csv
from functools import reduce
import torch
import torch.nn as nn
import copy

# Least common multiple
def lcm(a, b):
    return abs(a * b) // math.gcd(a, b)

def lcm_of_list(numbers):
    return reduce(lcm, numbers)


In [ ]:
# Shared scheduling state
class Shared:
    def __init__(self):
        self.slist = []
        self.is_primary_set = False


In [ ]:
class Task:
    def __init__(self, name, vf, initial_exec_task_time, deadline, alpha_c, I0_exp_term, peak_aC, min_aC, shared, WCET, worst_vf, exec_set, vf_levels, worst_lvl):
        self.name = name
        self.init_v, self.init_f = vf
        self.initial_exec_task_time = initial_exec_task_time
        self.current_exec_time = initial_exec_task_time
        self.alpha_C = alpha_c
        self.I0_exp_term = I0_exp_term
        self.peak_aC = peak_aC 
        self.min_aC = min_aC
        self.period = deadline
        self.deadline = deadline
        self.shared_tasks = shared
        self.arrival = 0
        self.fault = False
        self.WCET = WCET
        self.worst_v, self.worst_f = worst_vf
        self.exec_set = exec_set
        self.vf_levels = vf_levels
        self.v, self.f = vf
        self.core = None
        self.is_primary = False
        self.job_id = 0 
        self.slack = 0
        self.current_job_id = -1
        self.priority = -1

        self.worst_lvl = worst_lvl

    def get_util(self):
        return (self.current_exec_time/self.period)
    def update(self, v, f, core):
        self.f = f
        self.v = v
        self.core = core
        self.current_exec_time = math.ceil((self.initial_exec_task_time / (f/self.init_f)))
    
    def get_peak_p(self):
        return (self.peak_aC*(self.v**2)*self.f + self.I0_exp_term * self.v)
    def get_min_p(self):
        return (self.min_aC*(self.v**2)*self.f + self.I0_exp_term * self.v)
    def get_avg_p(self):
        return (self.alpha_C*(self.v**2)*self.f + self.I0_exp_term * self.v)
    
    def get_vf_lvl_peak(self, i):
        return self.peak_aC*(self.vf_levels[i]['voltage']**2)*self.vf_levels[i]['frequency'] + self.I0_exp_term * self.vf_levels[i]['voltage']


In [ ]:
class Job(Task):
    def __init__(self, task: Task, shared):
        super().__init__(
            name=task.name,
            vf=(task.init_v, task.init_f),
            initial_exec_task_time=task.initial_exec_task_time,
            deadline=task.deadline,
            alpha_c=task.alpha_C,
            I0_exp_term=task.I0_exp_term,
            peak_aC=task.peak_aC,
            min_aC=task.min_aC,
            shared=shared,
            WCET=task.WCET,
            worst_vf=(task.worst_v, task.worst_f),
            exec_set=task.exec_set,
            vf_levels=task.vf_levels,
            worst_lvl=task.worst_lvl
        )
        self.min_start_time = -1
        self.vf_is_set = False
        self.real_exec = 0
        self.is_real_exec_set = False
        self.start_time = -1
        self.finish_time = -1
        self.current_peak = -1
        self.reliability = -1
        self.real_reliability = -1

    def job_update(self, task: Task):
        self.task = task
        self.arrival = task.arrival
        self.fault = task.fault
        self.v, self.f = task.v, task.f
        self.priority = task.priority
        self.v_policy, self.f_policy = task.v, task.f
        self.core = task.core
        self.is_primary = task.is_primary
        self.current_exec_time = task.current_exec_time
        self.job_id = task.job_id

    def apply_DVFS(self, v, f):
        self.task.f = f
        self.task.v = v
        self.f = f
        self.v = v
        self.current_exec_time = math.ceil((self.task.initial_exec_task_time / (f/self.task.init_f)))
        self.task.current_exec_time = self.current_exec_time
    
    def get_current_util(self):
        return self.task.current_exec_time/self.task.period

    def get_current_peak(self):
        return (self.task.peak_aC*(self.task.v**2)*self.task.f + self.task.I0_exp_term * self.task.v)

    def get_rep_num(self):
        return (len(self.shared_tasks.slist) - 1)


In [ ]:
class Core:
    def __init__(self, id): 
        self.id = id
        self.ready = []
        self.running_replica = []
        self.mapped_for_slack = []
        self.primary = []
        self.replica_st = []
        self.replica_dl = []
        self.current_utilization = 0
        self.utilization = 0
        self.slack_time = 0
        self.running = None
    
    def NGSA_sort(self):
        self.primary.sort(key=lambda t:t.priority, reverse=True)
        self.replica_st.sort(key=lambda t:t.min_start_time)
        self.replica_dl.sort(key=lambda t:t.deadline)
    
    def remove_replica(self, j):
        exist = False 
        self.replica_dl.remove(j)
        self.replica_st.remove(j)
        self.ready.remove(j)
        if self.running == j:
            self.running=None
            exist = True
            # current_TDP[0] -= j.get_peak_p()
        return exist
    
    def remove_primary(self, j):
        self.primary.remove(j)
        self.ready.remove(j)
        if self.running == j:
            self.running=None
            # current_TDP[0] -= j.get_peak_p()


In [ ]:
# Quality-of-service metric

def utility(fi, Di, x=2):
    fi = np.asarray(fi, dtype=float)
    Di = np.asarray(Di, dtype=float)
    if x <= 1:
        raise ValueError("x must be > 1")
    UF = np.zeros_like(fi, dtype=float)
    on_time = fi <= Di
    soft = (fi > Di) & (fi <= x * Di)
    UF[on_time] = 1.0
    UF[soft] = (Di[soft] - fi[soft]) / (Di[soft] * (x - 1)) + 1.0
    return UF

def qos_system_no_backup(P, f_main, D, x=2):
    P = np.asarray(P, dtype=float)
    f_main = np.asarray(f_main, dtype=float)
    D = np.asarray(D, dtype=float)
    if not (len(P) == len(f_main) == len(D)):
        raise ValueError("P, f_main, and D must have the same length")
    if np.any((P < 0) | (P > 1)):
        raise ValueError("Probabilities must be in [0,1]")
    UF = utility(f_main, D, x)
    qos = np.sum(P * UF) / len(P)
    return float(qos), UF


In [ ]:
# Scheduling environment
class Envir(gym.Env):
    def __init__(self, initial_voltage, initial_frequency, cores, main_tasks, alpha):
        super().__init__()
        self.cores = cores
        self.main_tasks = main_tasks
        self.current_task_count = 0
        self.static = 0
        self.alpha = alpha
        self.data = []
        self.train_data = []
        # VF levels: frequency in Hz, voltage in V
        self.vf_levels = {
            0: {"frequency": 800e6, "voltage": 0.9},
            1: {"frequency": 1.1e9, "voltage": 0.95},
            2: {"frequency": 1.4e9, "voltage": 1.0},
            3: {"frequency": 1.7e9, "voltage": 1.05},
            4: {"frequency": 2.0e9, "voltage": 1.1},
        }
        self.initial_voltage = self.vf_levels[len(self.vf_levels)-1]['voltage']
        self.initial_frequency = self.vf_levels[len(self.vf_levels)-1]['frequency']
        self.lambda_0 = 1e-7  # faults/usec
        self.Delta = 1.0      # volt
        self.V_max = self.vf_levels[4]['voltage']  # max voltage

        self.system_tasks = []
        self.state = None
        self.action = None
        self.total_util = 0

        self.reset()

    def reset(self):
        self.state = {core: 0.0 for core in range(self.cores)}
        self.action = None
        self.total_util = 0
        self.train_data = self.data.copy()
        self.current_task_count = 0
        self.system_tasks.clear()
        return self.state

    def read_data(self, input_files):
        ds = []
        try:
            # def __init__(self, name, vf, initial_exec_task_time, deadline, alpha_c, I0_exp_term, peak_aC, static, min_aC):
            for file in input_files:
                
                task_name = os.path.basename(file).replace('.csv', '')
                task_data = pd.read_csv(file)
                
                initial_exec_task_time = task_data["#"].iloc[0]
                self.static = task_data["static"].iloc[0]
                total_power = task_data["avg_power"].iloc[0]
                alpha_C = task_data["alpha_C"].iloc[0]
                I0_exp_term = task_data["I0_exp_term"].iloc[0]
                util = task_data["util"].iloc[0]
                peak = task_data["peak"].iloc[0]
                deadline = task_data["deadline"].iloc[0]
                exec_set = {i:0 for i in range(len(self.vf_levels))}
                peak_list = []
                worst_lvl = -1
                for i in range(len(self.vf_levels)):
                    exec_set[i] = math.ceil(initial_exec_task_time / (self.vf_levels[i]['frequency']/self.initial_frequency))
                    peak_list.append((peak*(self.vf_levels[i]['voltage']**2)*self.vf_levels[i]['frequency'] + I0_exp_term*self.vf_levels[i]['voltage'], i))
                peak_list.sort(key=lambda t:t[0], reverse=True)
                for tup in peak_list:
                    if tup[0] <= TSP:
                        worst_lvl = tup[1]
                        break
                WCET = math.ceil(initial_exec_task_time / (self.vf_levels[worst_lvl]['frequency']/self.initial_frequency))
                
                #     exec_set[i] = math.ceil(initial_exec_task_time / (self.vf_levels[i]['frequency']/self.initial_frequency))
                self.data.append(Task(task_name, (self.initial_voltage, self.initial_frequency), 
                            initial_exec_task_time, deadline, alpha_C, I0_exp_term, peak, 
                            task_data["min"].head(1).mean(), Shared(), WCET, (self.vf_levels[worst_lvl]['voltage'], self.vf_levels[worst_lvl]['frequency']),
                            exec_set, self.vf_levels, worst_lvl
                            ))

            #     ds.append(z.deadline)
        except Exception as e:
            print(f"An error occurred: {e}")


In [ ]:
# Slack calculation
def calculate_slack(cores, hyper_period):
    for c in cores:
        total_WCET = 0
        for job in c.mapped_for_slack:
            total_WCET += job.WCET
        c.slack_time = hyper_period - total_WCET

def define_slack(cores, data, hyper_period):
    for c in cores:
        w_t = []
        for d in data:
            if c != d.core:
                continue
            w_t.append((hyper_period/d.period)*d.WCET)
        w_t = sum(w_t)
        for d in data:
            if c != d.core:
                continue
            d.slack = int((((hyper_period/d.period)*d.WCET)/w_t)*c.slack_time)


In [ ]:
# ============================================================
# NSGA-III mapper + VF selector + per-core priority assignment
# Add this AFTER Core / Task / Envir definitions.
#     map(data, ustate, env, cores)
# with:
#     nsga3_map(data, env, cores)
# ============================================================

import math
import random
import numpy as np

# ------------------------------------------------------------
# Basic helpers
# ------------------------------------------------------------

def _allowed_max_lvl(task, env):
    """
    Allowed VF levels are 0..task.worst_lvl.
    This preserves the TSP-based VF constraint.
    """
    max_lvl = getattr(task, "worst_lvl", len(env.vf_levels) - 1)

    if max_lvl is None or max_lvl < 0:
        max_lvl = len(env.vf_levels) - 1

    return min(max_lvl, len(env.vf_levels) - 1)

def _task_util_at_lvl(task, lvl):
    return task.exec_set[lvl] / task.period

def _decode_solution(solution, tasks, env, ncores):
    """
    Solution vector:
        first N genes -> core assignment
        next  N genes -> VF level
    """
    n = len(tasks)

    core_assign = []
    vf_assign = []

    for i, task in enumerate(tasks):
        c = int(round(solution[i]))
        c = max(0, min(ncores - 1, c))
        core_assign.append(c)

        max_lvl = _allowed_max_lvl(task, env)

        lvl = int(round(solution[n + i]))
        lvl = max(0, min(max_lvl, lvl))
        vf_assign.append(lvl)

    return core_assign, vf_assign

def _encode_solution(core_assign, vf_assign):
    return np.array(core_assign + vf_assign, dtype=float)

def _calc_core_utils(tasks, core_assign, vf_assign, ncores):
    utils = [0.0 for _ in range(ncores)]

    for task, c, lvl in zip(tasks, core_assign, vf_assign):
        utils[c] += _task_util_at_lvl(task, lvl)

    return utils

# ------------------------------------------------------------
# Priority assignment helper
# ------------------------------------------------------------

def _assign_priorities_per_core(tasks, core_assign, cores):
    """
    Assign priority separately inside each core.

    Rule:
      - For each core, tasks are ordered by the order they appear in the
        final best solution / mapped queue.
      - The task mapped earlier gets higher priority.
      - Higher priority is represented by a larger integer.

    Example:
      If core 0 has 4 tasks in mapped order:
          first  -> priority 4
          second -> priority 3
          third  -> priority 2
          fourth -> priority 1
    """

    tasks_by_core = {c.id: [] for c in cores}

    for task, core_id in zip(tasks, core_assign):
        tasks_by_core[core_id].append(task)

    for core in cores:
        q = tasks_by_core[core.id]
        n = len(q)

        for idx, task in enumerate(q):
            task.priority = n - idx

# ------------------------------------------------------------
# Repair solution to satisfy utilization <= 1.0 if possible
# ------------------------------------------------------------

def _repair_solution(tasks, core_assign, vf_assign, env, ncores):
    """
    Repair strategy:
      1. If a core is overloaded, increase VF level of tasks on it.
      2. If still overloaded, move tasks to less-loaded cores.
      3. Keep every task's VF level inside 0..task.worst_lvl.
    """
    core_assign = list(core_assign)
    vf_assign = list(vf_assign)

    for _ in range(10_000):
        utils = _calc_core_utils(tasks, core_assign, vf_assign, ncores)

        overloaded = [c for c in range(ncores) if utils[c] > 1.0 + 1e-12]

        if not overloaded:
            return core_assign, vf_assign

        changed = False

        # Step 1: speed up one task on overloaded cores
        best_change = None

        for c in overloaded:
            for i, task in enumerate(tasks):
                if core_assign[i] != c:
                    continue

                old_lvl = vf_assign[i]
                max_lvl = _allowed_max_lvl(task, env)

                if old_lvl >= max_lvl:
                    continue

                new_lvl = old_lvl + 1

                old_u = _task_util_at_lvl(task, old_lvl)
                new_u = _task_util_at_lvl(task, new_lvl)

                reduction = old_u - new_u

                if best_change is None or reduction > best_change[0]:
                    best_change = (reduction, i, new_lvl)

        if best_change is not None:
            _, task_idx, new_lvl = best_change
            vf_assign[task_idx] = new_lvl
            changed = True

        if changed:
            continue

        # Step 2: move one task from overloaded core to another core
        utils = _calc_core_utils(tasks, core_assign, vf_assign, ncores)
        overloaded = [c for c in range(ncores) if utils[c] > 1.0 + 1e-12]

        best_move = None

        for src in overloaded:
            for i, task in enumerate(tasks):
                if core_assign[i] != src:
                    continue

                current_lvl = vf_assign[i]
                current_u = _task_util_at_lvl(task, current_lvl)
                max_lvl = _allowed_max_lvl(task, env)

                for dst in range(ncores):
                    if dst == src:
                        continue

                    for new_lvl in range(current_lvl, max_lvl + 1):
                        new_u = _task_util_at_lvl(task, new_lvl)

                        src_after = utils[src] - current_u
                        dst_after = utils[dst] + new_u

                        if dst_after <= 1.0 + 1e-12:
                            overload_before = max(0.0, utils[src] - 1.0)
                            overload_after = max(0.0, src_after - 1.0)

                            improvement = overload_before - overload_after

                            if best_move is None or improvement > best_move[0]:
                                best_move = (improvement, i, dst, new_lvl)

        if best_move is not None:
            _, task_idx, dst, new_lvl = best_move
            core_assign[task_idx] = dst
            vf_assign[task_idx] = new_lvl
            changed = True

        if not changed:
            break

    return core_assign, vf_assign

# ------------------------------------------------------------
# NSGA-III objective functions
# ------------------------------------------------------------

def _evaluate_solution(solution, tasks, env, ncores):
    """
    NSGA-III minimizes all objectives.

    Objectives:
      f1: overload penalty
          -> must be 0 for valid mapping

      f2: distance from utilization 1.0
          -> makes each core close to full utilization

      f3: imbalance between cores
          -> balances load among cores

      f4: average VF level
          -> prevents always selecting unnecessarily high VF level
    """
    core_assign, vf_assign = _decode_solution(solution, tasks, env, ncores)

    core_assign, vf_assign = _repair_solution(
        tasks,
        core_assign,
        vf_assign,
        env,
        ncores
    )

    utils = _calc_core_utils(tasks, core_assign, vf_assign, ncores)

    overload = sum(max(0.0, u - 1.0) ** 2 for u in utils)

    distance_to_one = sum((1.0 - min(u, 1.0)) ** 2 for u in utils)

    mean_u = sum(utils) / ncores
    imbalance = sum((u - mean_u) ** 2 for u in utils) / ncores

    vf_cost = 0.0

    for task, lvl in zip(tasks, vf_assign):
        max_lvl = max(1, _allowed_max_lvl(task, env))
        vf_cost += lvl / max_lvl

    vf_cost /= max(1, len(tasks))

    return np.array(
        [
            overload,
            distance_to_one,
            imbalance,
            vf_cost,
        ],
        dtype=float
    )

def _constraint_violation(solution, tasks, env, ncores):
    core_assign, vf_assign = _decode_solution(solution, tasks, env, ncores)

    core_assign, vf_assign = _repair_solution(
        tasks,
        core_assign,
        vf_assign,
        env,
        ncores
    )

    utils = _calc_core_utils(tasks, core_assign, vf_assign, ncores)

    return sum(max(0.0, u - 1.0) for u in utils)

# ------------------------------------------------------------
# NSGA-III reference directions
# ------------------------------------------------------------

def _generate_reference_directions(n_obj, divisions):
    """
    Das-Dennis style reference directions.
    """
    ref_dirs = []

    def recursive_gen(left, total, depth, current):
        if depth == n_obj - 1:
            ref_dirs.append(current + [left / total])
        else:
            for i in range(left + 1):
                recursive_gen(
                    left - i,
                    total,
                    depth + 1,
                    current + [i / total]
                )

    recursive_gen(divisions, divisions, 0, [])

    ref_dirs = np.array(ref_dirs, dtype=float)

    norms = np.linalg.norm(ref_dirs, axis=1, keepdims=True)
    norms[norms < 1e-12] = 1.0

    ref_dirs = ref_dirs / norms

    return ref_dirs

# ------------------------------------------------------------
# Dominance and non-dominated sorting
# ------------------------------------------------------------

def _dominates(obj_a, cv_a, obj_b, cv_b):
    """
    Feasibility-first dominance:
      1. Feasible solution dominates infeasible solution.
      2. Between infeasible solutions, smaller violation is better.
      3. Between feasible solutions, use Pareto dominance.
    """
    eps = 1e-12

    a_feasible = cv_a <= eps
    b_feasible = cv_b <= eps

    if a_feasible and not b_feasible:
        return True

    if not a_feasible and b_feasible:
        return False

    if not a_feasible and not b_feasible:
        return cv_a < cv_b

    return np.all(obj_a <= obj_b + eps) and np.any(obj_a < obj_b - eps)

def _fast_non_dominated_sort(objectives, violations):
    pop_size = len(objectives)

    S = [[] for _ in range(pop_size)]
    n = [0 for _ in range(pop_size)]
    fronts = [[]]

    for p in range(pop_size):
        for q in range(pop_size):
            if p == q:
                continue

            if _dominates(objectives[p], violations[p], objectives[q], violations[q]):
                S[p].append(q)
            elif _dominates(objectives[q], violations[q], objectives[p], violations[p]):
                n[p] += 1

        if n[p] == 0:
            fronts[0].append(p)

    i = 0

    while fronts[i]:
        next_front = []

        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1

                if n[q] == 0:
                    next_front.append(q)

        i += 1
        fronts.append(next_front)

    if not fronts[-1]:
        fronts.pop()

    return fronts

# ------------------------------------------------------------
# NSGA-III environmental selection
# ------------------------------------------------------------

def _normalize_objectives(front_objs, all_objs):
    ideal = np.min(all_objs, axis=0)
    shifted = front_objs - ideal

    max_val = np.max(all_objs - ideal, axis=0)
    max_val[max_val < 1e-12] = 1.0

    normalized = shifted / max_val

    return normalized

def _associate_to_reference_dirs(norm_objs, ref_dirs):
    """
    Associate each solution with nearest reference direction.
    """
    associations = []
    distances = []

    for obj in norm_objs:
        norm = np.linalg.norm(obj)

        if norm < 1e-12:
            associations.append(0)
            distances.append(0.0)
            continue

        unit_obj = obj / norm

        cosine = np.dot(ref_dirs, unit_obj)
        cosine = np.clip(cosine, -1.0, 1.0)

        perpendicular_dist = np.sqrt(1.0 - cosine ** 2)

        ref_idx = int(np.argmin(perpendicular_dist))

        associations.append(ref_idx)
        distances.append(float(perpendicular_dist[ref_idx]))

    return associations, distances

def _nsga3_select(population, objectives, violations, pop_size, ref_dirs):
    fronts = _fast_non_dominated_sort(objectives, violations)

    selected = []
    last_front = []

    for front in fronts:
        if len(selected) + len(front) <= pop_size:
            selected.extend(front)
        else:
            last_front = front
            break

    if len(selected) == pop_size:
        return population[selected], objectives[selected], violations[selected]

    remaining = pop_size - len(selected)

    selected_objs = (
        objectives[selected]
        if selected
        else np.empty((0, objectives.shape[1]))
    )

    last_objs = objectives[last_front]

    all_considered = (
        objectives[selected + last_front]
        if selected
        else objectives[last_front]
    )

    norm_last = _normalize_objectives(last_objs, all_considered)

    last_assoc, last_dist = _associate_to_reference_dirs(norm_last, ref_dirs)

    niche_count = {i: 0 for i in range(len(ref_dirs))}

    if selected:
        norm_selected = _normalize_objectives(selected_objs, all_considered)
        selected_assoc, _ = _associate_to_reference_dirs(norm_selected, ref_dirs)

        for a in selected_assoc:
            niche_count[a] += 1

    candidates_by_ref = {i: [] for i in range(len(ref_dirs))}

    for local_idx, global_idx in enumerate(last_front):
        ref_idx = last_assoc[local_idx]
        dist = last_dist[local_idx]
        candidates_by_ref[ref_idx].append((global_idx, dist))

    for ref_idx in candidates_by_ref:
        candidates_by_ref[ref_idx].sort(key=lambda x: x[1])

    chosen = []

    while len(chosen) < remaining:
        available_refs = [
            r for r in range(len(ref_dirs))
            if len(candidates_by_ref[r]) > 0
        ]

        if not available_refs:
            break

        min_niche = min(niche_count[r] for r in available_refs)
        refs = [r for r in available_refs if niche_count[r] == min_niche]

        ref = random.choice(refs)

        if niche_count[ref] == 0:
            candidate, _ = candidates_by_ref[ref].pop(0)
        else:
            rand_idx = random.randrange(len(candidates_by_ref[ref]))
            candidate, _ = candidates_by_ref[ref].pop(rand_idx)

        chosen.append(candidate)
        niche_count[ref] += 1

    selected.extend(chosen)

    return population[selected], objectives[selected], violations[selected]

# ------------------------------------------------------------
# Initialization
# ------------------------------------------------------------

def _make_random_solution(tasks, env, ncores):
    core_assign = []
    vf_assign = []

    for task in tasks:
        core_assign.append(random.randint(0, ncores - 1))
        vf_assign.append(random.randint(0, _allowed_max_lvl(task, env)))

    core_assign, vf_assign = _repair_solution(
        tasks,
        core_assign,
        vf_assign,
        env,
        ncores
    )

    return _encode_solution(core_assign, vf_assign)

def _make_greedy_solution(tasks, env, ncores):
    """
    Good seed:
      - high-utilization tasks first
      - assign to the best core
      - choose lowest VF level that keeps utilization feasible if possible
    """
    n = len(tasks)

    core_assign = [0 for _ in range(n)]
    vf_assign = [0 for _ in range(n)]

    utils = [0.0 for _ in range(ncores)]

    indexed_tasks = list(enumerate(tasks))
    indexed_tasks.sort(
        key=lambda x: _task_util_at_lvl(x[1], 0),
        reverse=True
    )

    for i, task in indexed_tasks:
        best = None

        for c in range(ncores):
            for lvl in range(0, _allowed_max_lvl(task, env) + 1):
                u = _task_util_at_lvl(task, lvl)
                new_u = utils[c] + u

                overload = max(0.0, new_u - 1.0)

                score = (
                    overload,
                    abs(1.0 - min(new_u, 1.0)),
                    utils[c]
                )

                if best is None or score < best[0]:
                    best = (score, c, lvl, u)

        _, c, lvl, u = best

        core_assign[i] = c
        vf_assign[i] = lvl
        utils[c] += u

    core_assign, vf_assign = _repair_solution(
        tasks,
        core_assign,
        vf_assign,
        env,
        ncores
    )

    return _encode_solution(core_assign, vf_assign)

# ------------------------------------------------------------
# Genetic operators
# ------------------------------------------------------------

def _binary_tournament(population, objectives, violations):
    i = random.randrange(len(population))
    j = random.randrange(len(population))

    if _dominates(objectives[i], violations[i], objectives[j], violations[j]):
        return population[i].copy()

    if _dominates(objectives[j], violations[j], objectives[i], violations[i]):
        return population[j].copy()

    if random.random() < 0.5:
        return population[i].copy()

    return population[j].copy()

def _crossover(parent1, parent2, crossover_prob=0.9):
    if random.random() > crossover_prob:
        return parent1.copy(), parent2.copy()

    n = len(parent1)

    child1 = parent1.copy()
    child2 = parent2.copy()

    for i in range(n):
        if random.random() < 0.5:
            child1[i] = parent2[i]
            child2[i] = parent1[i]

    return child1, child2

def _mutate(solution, tasks, env, ncores, mutation_prob=None):
    n = len(tasks)
    dim = 2 * n

    if mutation_prob is None:
        mutation_prob = 1.0 / dim

    mutated = solution.copy()

    for i in range(dim):
        if random.random() > mutation_prob:
            continue

        if i < n:
            mutated[i] = random.randint(0, ncores - 1)
        else:
            task_idx = i - n
            max_lvl = _allowed_max_lvl(tasks[task_idx], env)
            mutated[i] = random.randint(0, max_lvl)

    core_assign, vf_assign = _decode_solution(
        mutated,
        tasks,
        env,
        ncores
    )

    core_assign, vf_assign = _repair_solution(
        tasks,
        core_assign,
        vf_assign,
        env,
        ncores
    )

    return _encode_solution(core_assign, vf_assign)

# ------------------------------------------------------------
# Main NSGA-III mapper
# ------------------------------------------------------------

def nsga3_map(
    data,
    env,
    cores,
    pop_size=92,
    generations=250,
    ref_divisions=4,
    crossover_prob=0.9,
    mutation_prob=None,
    seed=None,
    verbose=True
):
    """
    NSGA-III based task mapping + VF selection + priority assignment.

    Conditions:
      1. Map all tasks among cores.
      2. Balance load among cores.
      3. Select VF level per task.
      4. Try to make every core utilization close to 1.0.
      5. Enforce core utilization <= 1.0 if feasible.
      6. Enforce VF level <= task.worst_lvl.
      7. Assign task.priority separately inside each core.
         Earlier mapped task in that core gets higher priority.
    """

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    tasks = list(data)
    ncores = env.cores
    n_obj = 4

    ref_dirs = _generate_reference_directions(n_obj, ref_divisions)

    if pop_size < len(ref_dirs):
        pop_size = len(ref_dirs)

    # Clear previous mappings
    for c in cores:
        c.ready.clear()
        c.running_replica.clear()
        c.mapped_for_slack.clear()
        c.primary.clear()
        c.replica_st.clear()
        c.replica_dl.clear()
        c.current_utilization = 0.0
        c.utilization = 0.0
        c.slack_time = 0
        c.running = None

    for task in tasks:
        task.core = None
        task.is_primary = False
        task.shared_tasks.slist.clear()
        task.priority = -1

    # Initial population
    population = []

    population.append(_make_greedy_solution(tasks, env, ncores))

    while len(population) < pop_size:
        population.append(_make_random_solution(tasks, env, ncores))

    population = np.array(population, dtype=float)

    objectives = np.array(
        [
            _evaluate_solution(ind, tasks, env, ncores)
            for ind in population
        ],
        dtype=float
    )

    violations = np.array(
        [
            _constraint_violation(ind, tasks, env, ncores)
            for ind in population
        ],
        dtype=float
    )

    # Evolution loop
    for gen in range(generations):
        offspring = []

        while len(offspring) < pop_size:
            p1 = _binary_tournament(population, objectives, violations)
            p2 = _binary_tournament(population, objectives, violations)

            c1, c2 = _crossover(p1, p2, crossover_prob)

            c1 = _mutate(c1, tasks, env, ncores, mutation_prob)
            c2 = _mutate(c2, tasks, env, ncores, mutation_prob)

            offspring.append(c1)

            if len(offspring) < pop_size:
                offspring.append(c2)

        offspring = np.array(offspring, dtype=float)

        off_objectives = np.array(
            [
                _evaluate_solution(ind, tasks, env, ncores)
                for ind in offspring
            ],
            dtype=float
        )

        off_violations = np.array(
            [
                _constraint_violation(ind, tasks, env, ncores)
                for ind in offspring
            ],
            dtype=float
        )

        combined_pop = np.vstack([population, offspring])
        combined_obj = np.vstack([objectives, off_objectives])
        combined_cv = np.concatenate([violations, off_violations])

        population, objectives, violations = _nsga3_select(
            combined_pop,
            combined_obj,
            combined_cv,
            pop_size,
            ref_dirs
        )

    # Pick final solution from first non-dominated front
    fronts = _fast_non_dominated_sort(objectives, violations)
    first_front = fronts[0]

    best_idx = None
    best_score = None

    for idx in first_front:
        solution = population[idx]

        core_assign, vf_assign = _decode_solution(
            solution,
            tasks,
            env,
            ncores
        )

        core_assign, vf_assign = _repair_solution(
            tasks,
            core_assign,
            vf_assign,
            env,
            ncores
        )

        utils = _calc_core_utils(tasks, core_assign, vf_assign, ncores)

        overload = sum(max(0.0, u - 1.0) for u in utils)

        distance_to_one = sum(
            (1.0 - min(u, 1.0)) ** 2
            for u in utils
        )

        mean_u = sum(utils) / ncores

        imbalance = sum(
            (u - mean_u) ** 2
            for u in utils
        ) / ncores

        spread = max(utils) - min(utils)

        score = (
            overload,
            distance_to_one,
            imbalance,
            spread
        )

        if best_score is None or score < best_score:
            best_score = score
            best_idx = idx

    best_solution = population[best_idx]

    best_core_assign, best_vf_assign = _decode_solution(
        best_solution,
        tasks,
        env,
        ncores
    )

    best_core_assign, best_vf_assign = _repair_solution(
        tasks,
        best_core_assign,
        best_vf_assign,
        env,
        ncores
    )

    final_utils = _calc_core_utils(
        tasks,
        best_core_assign,
        best_vf_assign,
        ncores
    )

    # Assign priority before applying the final mapped order.
    _assign_priorities_per_core(tasks, best_core_assign, cores)

    # Apply final mapping to the Task/Core objects
    for task, core_id, lvl in zip(tasks, best_core_assign, best_vf_assign):
        vf = env.vf_levels[lvl]

        v = vf["voltage"]
        f = vf["frequency"]

        core = cores[core_id]

        task.update(v, f, core)
        task.is_primary = True
        task.shared_tasks.slist.append(task)

        core.current_utilization += task.get_util()
        core.utilization = core.current_utilization

        # Optional: keep mapped task queue inside each core as well
        core.mapped_for_slack.append(task)
    verbose = False
    if verbose:
        print("========== NSGA-III Mapping Result ==========")

        for c in cores:
            print(f"Core {c.id}: utilization = {c.current_utilization:.6f}")

            mapped_names = [
                f"{t.name}(priority={t.priority})"
                for t in c.mapped_for_slack
            ]

            print(f"Core {c.id} mapped queue: {mapped_names}")

        print("---------------------------------------------")

        for task in tasks:
            lvl = None

            for i, vf in env.vf_levels.items():
                if vf["frequency"] == task.f and vf["voltage"] == task.v:
                    lvl = i
                    break

            print(
                f"{task.name}: "
                f"core={task.core.id}, "
                f"priority={task.priority}, "
                f"vf_level={lvl}, "
                f"V={task.v}, "
                f"F={task.f}, "
                f"exec={task.current_exec_time}, "
                f"util={task.get_util():.6f}"
            )

        print("=============================================")

        if any(c.current_utilization > 1.0 + 1e-9 for c in cores):
            print("WARNING: NSGA-III could not find a fully feasible mapping.")
            print("At least one core utilization is still above 1.0.")
            print("This usually means the task set is infeasible with the available cores/VF levels.")

    return best_core_assign, best_vf_assign, final_utils


In [ ]:
# Mapping and job creation
def map(data, ustate, env, cores):
    """
    NSGA-III version of the mapper.

    It updates each task with:
        task.core
        task.v
        task.f
        task.current_exec_time

    It also updates:
        ustate
        core.current_utilization
        task.shared_tasks.slist

    This keeps create_jobs(...), calculate_slack(...), define_slack(...),
    dispatch(...), DVFS(...), and simulate(...) compatible with the code.
    """

    # Reset cores
    for c in cores:
        c.ready.clear()
        c.running_replica.clear()
        c.mapped_for_slack.clear()
        c.primary.clear()
        c.replica_st.clear()
        c.replica_dl.clear()
        c.current_utilization = 0.0
        c.utilization = 0.0
        c.slack_time = 0
        c.running = None

    # Reset ustate
    for k in list(ustate.keys()):
        ustate[k] = 0.0

    # Reset task mapping state
    for task in data:
        task.shared_tasks.slist.clear()
        task.core = None
        task.is_primary = False

    best_solution, best_util, best_obj = nsga3_optimize_mapping(
        data=data,
        env=env,
        population_size=92,
        generations=150,
        divisions=12,
        crossover_rate=0.9,
        mutation_rate=None,
        seed=42
    )

    n_tasks = len(data)
    core_assignment, vf_assignment = _decode_solution(best_solution, n_tasks)

    for i, task in enumerate(data):
        core_id = int(core_assignment[i])
        lvl = int(vf_assignment[i])

        v = env.vf_levels[lvl]["voltage"]
        f = env.vf_levels[lvl]["frequency"]
        core = cores[core_id]

        task.update(v, f, core)
        task.is_primary = True

        task.shared_tasks.slist.append(task)

        util = task.get_util()
        ustate[core_id] += util
        core.current_utilization += util
        core.utilization = core.current_utilization

    print("NSGA-III final core utilization:")
    for c in cores:
        print(f"Core {c.id}: {c.current_utilization:.6f}")

    if any(c.current_utilization > 1.0 + 1e-9 for c in cores):
        print("WARNING: NSGA-III could not make all core utilizations <= 1.0")

    return cores

# Job creation
def create_jobs(data, hyper_period, cores, env):
    jobs = []
    arrivals = []
    for t in range(hyper_period):
        for task in data:
            if task.arrival == t:
                p_job = Job(task, Shared())
                p_job.job_update(task)
                p_job.shared_tasks.slist.append(p_job)
                p_job.job_id = task.job_id
                jobs.append(p_job)
                arrivals.append(task.arrival)
                p_job.core.mapped_for_slack.append(p_job)
                p_job.is_primary = True
                for rep in task.shared_tasks.slist:
                    if rep == task: continue
                    r_job = copy.copy(p_job)
                    r_job.task = rep
                    r_job.core = rep.core
                    r_job.min_start_time = (r_job.deadline - r_job.WCET)
                    r_job.is_primary = False
                    r_job.shared_tasks.slist.append(r_job)
                    jobs.append(r_job)
                
                task.arrival += task.period
                task.deadline += task.period
                task.job_id += 1
    arrivals = sorted(list(set(arrivals)))
    return jobs, arrivals
                
def bounded_normal(min_val, max_val, center, spread_factor=0.25):
    range_val = max_val - min_val
    std_dev = spread_factor * range_val
    a, b = (min_val - center) / std_dev, (max_val - center) / std_dev
    return truncnorm.rvs(a, b, loc=center, scale=std_dev)


In [ ]:
# Arrival and dispatch
def manage_arrival(arrival_queue, cores, job_queue, time):

    new_arrived = False
    arr = -1
    if arrival_queue: arr = arrival_queue.popleft()
    if time == arr:
        found = False
        while not found:
            if not job_queue:
                break
            job = job_queue.popleft()
            if job.arrival != arr:
                job_queue.appendleft(job)
                found = True
                break
            if not job.is_primary:
                job.core.replica_st.append(job)
                job.core.replica_dl.append(job)
                job.core.ready.append(job)
            else:
                job.core.primary.append(job)
                job.core.ready.append(job)
            new_arrived = True
    else: arrival_queue.appendleft(arr)
# is_replica
    return new_arrived

def logic_of_replicas(c, time):
    if c.replica_dl[0].deadline <= c.replica_st[0].deadline and c.replica_dl[0].min_start_time <= c.replica_st[0].min_start_time and c.replica_dl[0].min_start_time<=time:
        job = c.replica_dl[0]
        c.running = job
    elif c.replica_dl[0].deadline >= c.replica_st[0].deadline and c.replica_dl[0].min_start_time >= c.replica_st[0].min_start_time and c.replica_dl[0].min_start_time<=time:
        job = c.replica_st[0]
        c.running = job
    elif c.replica_dl[0].deadline < c.replica_st[0].deadline and c.replica_dl[0].min_start_time > c.replica_st[0].min_start_time and c.replica_st[0].min_start_time<=time:
        job = c.replica_st[0]
        c.running = job
    elif c.replica_dl[0].deadline > c.replica_st[0].deadline and c.replica_dl[0].min_start_time < c.replica_st[0].min_start_time and c.replica_dl[0].min_start_time<=time:
        job = c.replica_dl[0]
        c.running = job

def dispatch(cores, time):

    for c in cores:
        # both are empty
        if (not c.primary) and (not c.replica_dl): continue
        # already a job is running
        if c.running:
            # replicas is empty
            if c.primary and (not c.replica_dl):
                if c.primary[0].priority > c.running.priority:
                    c.running = c.primary[0]
            # primary is empty
            elif (not c.primary) and c.replica_dl:
                if c.replica_dl[0].deadline < c.running.deadline and c.replica_dl[0].min_start_time <= time:
                    logic_of_replicas(c, time)

            else:
                if c.primary[0].deadline < c.running.deadline:
                    c.running = c.primary[0]

                if c.replica_dl[0].deadline < c.running.deadline and c.replica_dl[0].min_start_time <= time:
                    logic_of_replicas(c, time)

        # nothin is running
        else:
            # replicas is empty
            if c.primary and (not c.replica_dl): 
                c.running = c.primary[0]
            # primary is empty
            elif (not c.primary) and c.replica_dl:
                logic_of_replicas(c, time)
            # both not empty
            elif (c.primary[0].deadline <= c.replica_dl[0].deadline) or (c.replica_dl[0].min_start_time > time and c.replica_st[0].min_start_time > time):
                job = c.primary[0]
                c.running = job
            else:
                logic_of_replicas(c, time)


In [ ]:
# Dynamic voltage and frequency scaling

def DVFS(cores, time, vf_levels):
    for c in cores:
        if (c.current_utilization< 0): print(time ,c.current_utilization)
    for c in cores:
        if (not c.running) or c.running.vf_is_set: continue
        worst_lvl = c.running.worst_lvl
        if not c.running.is_primary:
            c.running.apply_DVFS(c.running.worst_v, c.running.worst_f)
            u_new = c.running.get_current_util()
            c.current_utilization += u_new
            c.running.task.current_job_id = c.running.job_id
            c.running_replica.append(c.running.task)
    
        elif c.current_utilization > 1:
            
            new_c_util = []
            for i in range(worst_lvl+1):
                ne = c.current_utilization + (c.running.task.exec_set[i]/c.running.task.period - c.running.get_current_util())
                if ne <= 1 and ne >=0 :
                    new_c_util.append((ne, i))
            # core util <= 1
            if new_c_util:
                new_c_util.sort(key=lambda t:t[0], reverse=True)
                lvl = new_c_util[0][1]
                u = c.running.get_current_util()
                c.running.apply_DVFS(vf_levels[lvl]['voltage'], vf_levels[lvl]['frequency'])
                u_new = c.running.get_current_util()
                c.current_utilization += (u_new - u)
            # core util > 1
            # Update core utilization
            else:
                u = c.running.get_current_util()
                c.running.apply_DVFS(c.running.task.worst_v, c.running.task.worst_f)
                u_new = c.running.get_current_util()
                c.current_utilization += (u_new - u)
                   
        else:
            new_c_util = []
            for i in range(worst_lvl+1):
                ne = c.current_utilization + (c.running.task.exec_set[i]/c.running.task.period - c.running.get_current_util())
                if ne <= 1 and ne>=0:
                    new_c_util.append((ne, i))
            
            # core util <= 1
            
            for i in range(len(new_c_util)):
                if vf_levels[new_c_util[i][1]]['frequency'] == c.running.f and vf_levels[new_c_util[i][1]]['voltage'] == c.running.v:
                    c.running.apply_DVFS(c.running.v, c.running.f)
                    break
                new_c_util.sort(key=lambda t:t[0], reverse=True)
                lvl = new_c_util[i][1]
                slack_covered = (c.running.task.initial_exec_task_time / (vf_levels[lvl]['frequency']/c.running.task.init_f)) - c.running.task.current_exec_time
                if c.running.task.slack - slack_covered < 0:
                    continue
                c.running.task.slack -= slack_covered
                u = c.running.get_current_util()
                c.running.apply_DVFS(vf_levels[lvl]['voltage'], vf_levels[lvl]['frequency'])
                u_new = c.running.get_current_util()
                c.current_utilization += (u_new - u)
                break

        c.running.vf_is_set = True
        c.running.real_exec = c.running.current_exec_time
        c.running.is_real_exec_set = True


In [ ]:
# Execution and completion handling
def execute(cores, time, power_trace, util_trace, total_jobs, total_finished_jobs, faulty_jobs_counter, hyper_period):
    if total_jobs + faulty_jobs_counter == total_finished_jobs and time>=hyper_period: return
    u_tup = [0 for c in cores]
    for c in cores:
        if not c.running: continue # core is idle
        job = c.running
        if job.start_time == -1: job.start_time = time - 1
        job.real_exec -= 1

        if random.random() < 0.1:
            power_trace[c.id].append(job.get_peak_p())
        #     power_trace[c.id].append(p)
        else:    
            power_trace[c.id].append(bounded_normal(job.get_min_p(), job.get_peak_p(), job.get_avg_p()))
        u_tup[c.id] = (c.current_utilization)

    # power_trace.append(tuple(pt_tup))
    util_trace.append(tuple(u_tup))
    
def handle_finished_jobs(cores, time, TSP):
    TSP_violate_counter = 0
    total_finished_jobs = 0
    deadline_missed_counter = 0
    faulty_jobs_counter = 0

    utils = []
    for c in cores:
        utils.append(c.current_utilization)
        if (not c.running) or (c.running.real_exec>0): continue # core is idle or job is not done yet
        job = c.running
        job.finish_time = time

        if job.get_peak_p() > TSP:
            TSP_violate_counter += 1
        total_finished_jobs += 1

        if job.deadline < time:
            deadline_missed_counter += 1
        if job.fault:
            faulty_jobs_counter +=1
            if not job.is_primary:
                job.core.remove_replica(job)
                job.shared_tasks.slist.remove(job)
            else:
                job.core.remove_primary(job)
                job.shared_tasks.slist.remove(job)
            continue

        if job.is_primary:
            
            for rep in job.shared_tasks.slist:
                if rep.is_primary: continue
                exs = rep.core.remove_replica(rep)
                if exs:
                    pass
                else:
                    if (rep.task in rep.core.running_replica) and (rep.job_id > rep.task.current_job_id):
                        rep.core.running_replica.remove(rep.task)
                        rep.core.current_utilization -= rep.get_current_util()
            c.remove_primary(job)
            c.running = None
        else:
            for rep in job.shared_tasks.slist:
                if rep.is_primary: 
                    rep.core.remove_primary(rep)
                    continue
                exs = rep.core.remove_replica(rep)
                if exs:
                    pass
                else:
                    if (rep.task in rep.core.running_replica) and (rep.job_id > rep.task.current_job_id):
                        rep.core.running_replica.remove(rep.task)
                        rep.core.current_utilization -= rep.get_current_util()

    return TSP_violate_counter, total_finished_jobs, deadline_missed_counter, faulty_jobs_counter

def inject_fault(cores):
    available_core = []
    for c in cores:
        if c.ready:
            available_core.append(c)
    if not available_core: return
    c = random.choice(available_core)

    job = random.choice(c.ready)
    job.fault = True


In [ ]:
def simulate(env, data, TSP, hyper_period, output_path, log_path, total_jobs):
    cores = [Core(c) for c in range(env.cores)]
    data = sorted(data, key=lambda d:d.get_util(), reverse=True)
    nsga3_map(
        data=data,
        env=env,
        cores=cores,
        pop_size=92,
        generations=250,
        ref_divisions=4,
        seed=42,
        verbose=True
    )
    for c in cores:
        c.ready.clear()
        c.running_replica.clear()
        c.mapped_for_slack.clear()
        c.primary.clear()
        c.replica_st.clear()
        c.replica_dl.clear()
        c.slack_time = 0
        c.running = None
    print(f"{cores[0].current_utilization}, {cores[1].current_utilization}, {cores[1].current_utilization}, {cores[3].current_utilization}")

    system = []
    #     system.append(total_task_reliability(tup))
    
    jobs, arrivals = create_jobs(data, hyper_period, cores, env)

# Calculate slack time
    calculate_slack(cores, hyper_period)
    define_slack(cores, data, hyper_period)

    #     # print(j.name, j.job_id, j.task)
    #         slacks.append(j.slack)
    
    arrival_queue = deque(arrivals)

    job_queue = deque(sorted(jobs, key=lambda j:(j.arrival, j.deadline)))

    new_arrived = False
    power_trace = [[] for c in cores]
    util_trace = []
    slack_trace = [[] for c in cores]
    TSP_violate_counter = 0
    total_finished_jobs = 0
    deadline_missed_counter = 0
    faulty_jobs_counter = 0
    do_fault = -1

# Runtime simulation
    for time in range(int(hyper_period*2 + 1)):

# Record per-core activity through the hyperperiod
        if not (total_jobs + faulty_jobs_counter == total_finished_jobs and time>=hyper_period):
            for c in cores: 
                if c.running:
                    slack_trace[c.id].append(1)
                else:
                    slack_trace[c.id].append(0)
        # # print(slack_trace)
        #         s_tr[c.id] = 1
        # slack_trace.append(s_tr)
        # # print(slack_trace)
        execute(cores, time, power_trace, util_trace, total_jobs, total_finished_jobs, faulty_jobs_counter, hyper_period)
        
        rv1, rv2, rv3, rv4 = handle_finished_jobs(cores, time, TSP)
        TSP_violate_counter += rv1
        total_finished_jobs += rv2
        deadline_missed_counter += rv3
        faulty_jobs_counter += rv4
        new_arrived = manage_arrival(arrival_queue, cores, job_queue, time)
        if new_arrived:
            for core in cores:
                core.NGSA_sort()

        dispatch(cores, time)
        DVFS(cores, time, env.vf_levels)
        utils = []

    print('\n=============================\n')
    print(TSP_violate_counter,
        total_finished_jobs,
        deadline_missed_counter,
        faulty_jobs_counter)
    print('\n=============================\n')
#     # for core in cores:
#     #     print(time, core.current_utilization, (core.running.name, core.running.job_id, 
#     #     core.running.rep_id, core.running.f, core.running.f_policy, core.running.worst_f,
#     #      core.running.init_f), core.running.real_exec, core.running.exec_set[0], core.running.current_exec_time,core.running.exec_set[4])
#     # print('\n----------------------------\n')

# # """
#     # print(arrivals)
    D_main =[]
    f_main =[]
    for d in jobs:
            if d.finish_time !=-1:
                D_main.append(d.deadline)
                f_main.append(d.finish_time)
    P = [1 for _ in range(len(f_main))]
    qos, UF = qos_system_no_backup(P, f_main, D_main)
                
    file_name = output_path + 'task_detail.csv'    
    with open(file_name, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Name", "Job_id","core_id","Arrival", "StartTime", "FinishTime", "Deadline", "Period", "v", "f", "reliability", "peak_p", "avg_p", "min_p"])
        for j in jobs:
            writer.writerow([j.name, j.job_id, j.core.id+1, j.arrival, j.start_time, j.finish_time, j.deadline, j.period, j.v, j.f, j.reliability, j.get_peak_p(), j.get_avg_p(), j.get_min_p()])

    energy = 0
    file_name = output_path + 'power_energy.csv'
    with open(file_name, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Core1 load(%)","Core1 peak","Core1 P", "Core1 E","Core2 load(%)","Core2 peak", "Core2 P", "Core2 E","Core3 load(%)","Core3 peak","Core3 P", "Core3 E","Core4 load(%)","Core4 peak","Core4 P", "Core4 E"])
        tr = []
        for c in cores:
            tr.append(round((sum(slack_trace[c.id])/len(slack_trace[c.id]))*100, 2))
            tr.append(max(power_trace[c.id]))
            avg = sum(power_trace[c.id])/len(power_trace[c.id])
            tr.append(avg)
            tr.append(avg*hyper_period*1e-3)
            energy += avg*hyper_period*1e-3
        writer.writerow(tr)
        print(tr)
    ENERGY = 0
    for j in jobs:
        if j.is_real_exec_set: ENERGY += (j.current_exec_time-j.real_exec) * (1e-3)* j.get_avg_p()

    file_name = output_path + 'information.csv'
    with open(file_name, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["ENERGY", "Energy" ,"QoS(%)"])
        writer.writerow([round(ENERGY, 2), round(energy, 2), round(qos*100, 3)])
        print([round(ENERGY, 2), round(energy, 2), round(qos*100, 3)])
        # writer.writerow([j.real_exec, j.current_exec_time, j.is_real_exec_set])
    # with open(file_name, mode='w', newline='') as file:
    #     writer.writerow(["Core 1", "Core 2", "Core 3", "Core 4"])
    #             tr.append(round(current_util[i], 3))
    #         writer.writerow(tr)

    # with open(file_name, mode='w', newline='') as file:
    #     writer.writerow(["ENERGY,Energy ,QoS(%)", "TSP Met Ratio(%)", "System reliability"])
    #     writer.writerow([round(qos*100, 3), round(((1-TSP_violate_counter/total_finished_jobs)*100), 3), system_r])
    # # return qos, TSP_violate_counter, total_finished_jobs, deadline_missed_counter, faulty_jobs_counter


In [ ]:
# Full baseline sweeps are opt-in because they can run for several hours.
RUN_BASELINE = False

if RUN_BASELINE:
    raise NotImplementedError(
        "Select load points from data/generated and call simulate(...) with an artifacts/ output path."
    )
